<a href="https://colab.research.google.com/github/M4rck0/Datos_Masivos/blob/main/Tarea_4-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import os, glob
from pyspark.sql import types as T
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, VectorAssembler
from pyspark.ml.classification import LogisticRegression

In [2]:
spark = (
    SparkSession.builder
    .appName("SparkEnColab")
    .master("local[*]")
    .config("spark.ui.enabled", "false") # Interfaz web
    .config("spark.driver.memory", "2g") # Limite RAM
    .getOrCreate() # Crear sesión
)

In [3]:
# Leer zip y descomprimir
zip_name = "reddit_technology.zip"
extract_dir = "reddit_extraido"
!unzip -q -o {zip_name} -d {extract_dir}

In [5]:
# Leer parquet
parquets = glob.glob(f"{extract_dir}/**/*.parquet", recursive=True)
ruta_parquet = os.path.commonpath([os.path.dirname(p) for p in parquets])
df = (spark.read
      .option("recursiveFileLookup", "true")
      .parquet(ruta_parquet)
)

# Limpieza (eliminados y nulos)
df = (df
    .filter(~((F.col("author") == "[deleted]") |
              (F.col("body").isin("[deleted]", "[removed]"))))
    .filter(F.col("score").isNotNull())
    .select("created_utc", "author", "score", "body")
)

df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+
|created_utc|       author|score|                                                                            body|
+-----------+-------------+-----+--------------------------------------------------------------------------------+
| 1432313203|      AbeRego|    3|                                 I read this in the cliché teen Simpson's voice.|
| 1432313210|   hefnetefne|    1|How about a law that says you can sue corporations, instead of proclaiming co...|
| 1432313243|newloginisnew|   23|Please share 100% of your browsing history as a comment reply. If you do not,...|
| 1432313244|       NeonHD|    1|My post wasn't supposed to be a question, it was a link which the original ti...|
| 1432313246|        Zoura|   47|As a former BBV employee I would just like to say, well done! Their crappy bu...|
+-----------+-------------+-----+-----------------------------------------------

In [6]:
# Manipulación columnas (seleccionar, renombrar y reordenar)
df = (df
       .withColumnRenamed("author", "autor")
       .withColumnRenamed("body", "mensaje")
       .select("created_utc", "autor", "score", "mensaje")
      )

df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+
|created_utc|        autor|score|                                                                         mensaje|
+-----------+-------------+-----+--------------------------------------------------------------------------------+
| 1432313203|      AbeRego|    3|                                 I read this in the cliché teen Simpson's voice.|
| 1432313210|   hefnetefne|    1|How about a law that says you can sue corporations, instead of proclaiming co...|
| 1432313243|newloginisnew|   23|Please share 100% of your browsing history as a comment reply. If you do not,...|
| 1432313244|       NeonHD|    1|My post wasn't supposed to be a question, it was a link which the original ti...|
| 1432313246|        Zoura|   47|As a former BBV employee I would just like to say, well done! Their crappy bu...|
+-----------+-------------+-----+-----------------------------------------------

In [7]:
# Convertir created_utc a timestamp (segundos a fecha)
df = (df
       .withColumn("created_utc", F.col("created_utc").cast("long"))
       .withColumn("fecha_ts", F.from_unixtime("created_utc").cast("timestamp"))
       .withColumn("score", F.col("score").cast("double"))
       .withColumn("mensaje", F.col("mensaje").cast("string"))
      )

df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+
|created_utc|        autor|score|                                                                         mensaje|           fecha_ts|
+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+
| 1432313203|      AbeRego|  3.0|                                 I read this in the cliché teen Simpson's voice.|2015-05-22 16:46:43|
| 1432313210|   hefnetefne|  1.0|How about a law that says you can sue corporations, instead of proclaiming co...|2015-05-22 16:46:50|
| 1432313243|newloginisnew| 23.0|Please share 100% of your browsing history as a comment reply. If you do not,...|2015-05-22 16:47:23|
| 1432313244|       NeonHD|  1.0|My post wasn't supposed to be a question, it was a link which the original ti...|2015-05-22 16:47:24|
| 1432313246|        Zoura| 47.0|As a former BBV employ

In [8]:
# Features simples
df = (df
       .withColumn("longitud_mensaje", F.length("mensaje"))
       .withColumn("num_palabras", F.size(F.split(F.col("mensaje"), r"\s+")))
       .withColumn("hora", F.hour("fecha_ts"))
      )

df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+----------------+------------+----+
|created_utc|        autor|score|                                                                         mensaje|           fecha_ts|longitud_mensaje|num_palabras|hora|
+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+----------------+------------+----+
| 1432313203|      AbeRego|  3.0|                                 I read this in the cliché teen Simpson's voice.|2015-05-22 16:46:43|              47|           9|  16|
| 1432313210|   hefnetefne|  1.0|How about a law that says you can sue corporations, instead of proclaiming co...|2015-05-22 16:46:50|             148|          25|  16|
| 1432313243|newloginisnew| 23.0|Please share 100% of your browsing history as a comment reply. If you do not,...|2015-05-22 16:47:23|             120

In [9]:
# Etiqueta binaria: popular si score >= percentil 90
p90 = df.approxQuantile("score", [0.90], 0.01)[0]
df = df.withColumn("etiqueta", F.when(F.col("score") >= F.lit(p90), 1.0).otherwise(0.0))
df.show(5, truncate=80)

+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+----------------+------------+----+--------+
|created_utc|        autor|score|                                                                         mensaje|           fecha_ts|longitud_mensaje|num_palabras|hora|etiqueta|
+-----------+-------------+-----+--------------------------------------------------------------------------------+-------------------+----------------+------------+----+--------+
| 1432313203|      AbeRego|  3.0|                                 I read this in the cliché teen Simpson's voice.|2015-05-22 16:46:43|              47|           9|  16|     0.0|
| 1432313210|   hefnetefne|  1.0|How about a law that says you can sue corporations, instead of proclaiming co...|2015-05-22 16:46:50|             148|          25|  16|     0.0|
| 1432313243|newloginisnew| 23.0|Please share 100% of your browsing history as a comment reply. If you do

In [10]:
# Conjunto de datos final para ML
df_ml = (df
         .select("etiqueta", "mensaje", "longitud_mensaje", "num_palabras", "hora")
         .dropna(subset=["etiqueta", "mensaje", "longitud_mensaje", "num_palabras", "hora"])
        )

df_ml.show(5, truncate=80)

+--------+--------------------------------------------------------------------------------+----------------+------------+----+
|etiqueta|                                                                         mensaje|longitud_mensaje|num_palabras|hora|
+--------+--------------------------------------------------------------------------------+----------------+------------+----+
|     0.0|                                 I read this in the cliché teen Simpson's voice.|              47|           9|  16|
|     0.0|How about a law that says you can sue corporations, instead of proclaiming co...|             148|          25|  16|
|     1.0|Please share 100% of your browsing history as a comment reply. If you do not,...|             120|          22|  16|
|     0.0|My post wasn't supposed to be a question, it was a link which the original ti...|             191|          38|  16|
|     1.0|As a former BBV employee I would just like to say, well done! Their crappy bu...|             156|   

In [11]:
df_ml.groupBy("etiqueta").count().show()
print("Percentil 90 del score =", p90)

+--------+------+
|etiqueta| count|
+--------+------+
|     0.0|160610|
|     1.0| 19468|
+--------+------+

Percentil 90 del score = 14.0


In [12]:
# Convierte el texto en una lista de palabras (tokens).
tokenizador = RegexTokenizer(
    inputCol="mensaje", # Columna
    outputCol="tokens",
    pattern=r"\W+"
)

# Quita palabras muy comunes (the, and, etc,...)
removedor_stopwords = StopWordsRemover(
    inputCol="tokens",
    outputCol="tokens_limpios"
)

# Convierte tokens en un vector numérico de frecuencias
tf = HashingTF(
    inputCol="tokens_limpios",
    outputCol="tf",
    numFeatures=1 << 18
)

# Baja el peso de palabras comunes y sube el de palabras raras
idf = IDF(
    inputCol="tf",
    outputCol="tfidf"
)

# Une todas tus variables en una sola columna caracteristicas
ensamblador = VectorAssembler(
    inputCols=["tfidf", "longitud_mensaje", "num_palabras", "hora"],
    outputCol="caracteristicas"
)

# Modelo Regresión Logística
modelo_lr = LogisticRegression(
    featuresCol="caracteristicas",
    labelCol="etiqueta",
    maxIter=20
)

# Consolida pasos
pipeline = Pipeline(stages=[
    tokenizador,
    removedor_stopwords,
    tf,
    idf,
    ensamblador,
    modelo_lr
])

In [13]:
# Split
entrenamiento, prueba = df_ml.randomSplit([0.7, 0.3], seed=42)

# Entrenar
modelo = pipeline.fit(entrenamiento)

# Predecir
predicciones = modelo.transform(prueba)
predicciones.select("etiqueta", "prediction", "probability").show(5, truncate=False)

# Accuracy
accuracy = predicciones.filter(F.col("etiqueta") == F.col("prediction")).count() / predicciones.count()

print("Accuracy =", accuracy)

+--------+----------+-------------------------------------------+
|etiqueta|prediction|probability                                |
+--------+----------+-------------------------------------------+
|0.0     |0.0       |[0.9923891924955309,0.007610807504469075]  |
|0.0     |0.0       |[0.9999999999999982,1.7763568394002505E-15]|
|0.0     |1.0       |[0.0019410373363536008,0.9980589626636464] |
|0.0     |0.0       |[0.8093468179681886,0.19065318203181136]   |
|0.0     |0.0       |[0.9996523392052791,3.476607947209276E-4]  |
+--------+----------+-------------------------------------------+
only showing top 5 rows
Accuracy = 0.8243714306904991


In [14]:
# Conteos
conteos = (predicciones
      .select(F.col("etiqueta").cast("int").alias("etiqueta"),
              F.col("prediction").cast("int").alias("prediccion"))
      .groupBy("etiqueta", "prediccion")
      .count()
)

# Totales por etiqueta real
totales = (conteos.groupBy("etiqueta")
           .agg(F.sum("count").alias("total_fila"))
)

# Porcentaje por fila
conteos_porcentaje = (conteos.join(totales, on="etiqueta", how="left")
          .withColumn("porcentaje_fila", F.col("count") / F.col("total_fila"))
          .orderBy("etiqueta", "prediccion")
)

conteos_porcentaje.show(truncate=False)

+--------+----------+-----+----------+-------------------+
|etiqueta|prediccion|count|total_fila|porcentaje_fila    |
+--------+----------+-----+----------+-------------------+
|0       |0         |43727|48156     |0.9080280754215466 |
|0       |1         |4429 |48156     |0.09197192457845337|
|1       |0         |5043 |5776      |0.873095567867036  |
|1       |1         |733  |5776      |0.12690443213296398|
+--------+----------+-----+----------+-------------------+



En este trabajo se construyó un modelo de clasificación binaria para predecir si un comentario en Reddit sería popular, definiendo como popular aquellos que pertenecen al percentil $90$ del score.

La variable objetivo presenta un fuerte desbalance de clases, ya que aproximadamente el $10\%$ de los comentarios corresponden a la clase positiva (popular). En este contexto, la exactitud (accuracy) puede resultar engañosa, por lo que es más apropiado evaluar métricas como precisión, sensibilidad (recall) y F1.

Aunque el modelo alcanzó una exactitud global de $82.4\%$, la matriz de confusión mostró que solo el $12.7\%$ de los comentarios populares fueron correctamente identificados (bajo recall). Esto indica que el modelo tiende a favorecer la clase mayoritaria (comentarios no populares).

Estos resultados sugieren que la popularidad en Reddit no depende únicamente del contenido textual o de variables simples como la hora o la longitud del mensaje, sino que probablemente intervienen factores adicionales como la reputación del autor, el contexto del hilo y la dinámica social de la comunidad.